# Project 1: Cybersecurity Alert Triage (SOC Operations)
**Synthetic Dataset Specification & Mathematical Blueprint**

---

| Item | Detail |
|:---|:---|
| **Author** | *Mark Christian Anub* |
| **Date** | *June 2026* |
| **Status** | Draft |
| **Target Learner** | SOC Analyst / Cybersecurity Data Scientist |
| **Tables** | 6 core tables, ~44 features |
| **Omitted Variables** | 2 latent variables (dropped before export) |

---

### Abstract
This document specifies the mathematical model, data schema, and structural equations for a synthetic cybersecurity dataset simulating **30 days** of Security Operations Center (SOC) telemetry. The generator produces realistic network logs, SIEM alerts, and analyst response data with **two hidden latent variables** that inject realistic confounding and label noise.

The resulting dataset challenges learners to build ML classification models that separate **True Positive breaches (~3%)** from **False Positive noise (~95%)** while navigating:
- Extreme class imbalance
- Missing values (~3–5% in key fields)
- Timestamp jitter (±3 seconds)
- Duplicate alerts (~5%)
- Human-induced label corruption from analyst fatigue

## 1. Business Context & Challenge Statement

Security Operations Centers (SOCs) suffer from severe **"Alert Fatigue."** Enterprise SIEMs (Security Information and Event Management systems) generate tens of thousands of alerts daily, of which **95–98% are false positives** caused by misconfigured rules or benign anomalies. Meanwhile, actual threat actors execute multi-stage attack chains (e.g., credential theft $\\rightarrow$ lateral movement $\\rightarrow$ data exfiltration) that hide within this noise.

### 1.1 The Learner's Challenge
You are a Data Scientist hired by a Fortune 500 company's SOC. You have been provided with **30 days** of messy network telemetry, SIEM alerts, and analyst response logs across **6 relational tables**. Your goal is to build a Machine Learning classification model to predict the `final_verdict` of an alert:
- **True Positive (TP):** A genuine cybersecurity breach requiring immediate incident response.
- **False Positive (FP):** A benign alert caused by misconfigured rules, routine scans, or non-malicious anomalies.
- **Benign True Positive (BTP):** A real but low-risk event (e.g., an authorized penetration test).

### 1.2 What Makes This Dataset Unique

> **⚠️ Warning to Learners:** This dataset contains realistic *human errors*. SOC analysts working long shifts occasionally mislabel true breaches as false positives due to cognitive fatigue. Your model must learn the "digital fingerprints" of an attack while navigating noisy, missing, and mislabeled data. Simple models that trust the labels blindly **will fail**.

## 2. Data Schema

The synthetic environment consists of **6 relational tables** with **~44 total features** and **2 hidden latent variables** (dropped before export).

### 2.1 Schema Overview

| # | Table | PK | FK(s) | Scale | Purpose |
|:---|:---|:---|:---|:---|:---|
| 1 | `users` | `user_id` | — | ~200–500 | Employee directory & behavioral baselines |
| 2 | `devices` | `device_id` | `user_id` | ~300–800 | Network assets & criticality ratings |
| 3 | `log_events` | `event_id` | `device_id`, `user_id` | ~500K–5M | High-volume time-series telemetry |
| 4 | `alerts` | `alert_id` | `device_id` | ~10K–50K | SIEM-aggregated security alerts |
| 5 | `response_actions` | `action_id` | `alert_id` | ~10K–50K | SOC analyst triage actions & shift metadata |
| 6 | `alert_outcomes` | `alert_id` | `alert_id` | ~10K–50K | Ground-truth labels & financial impact |

### 2.2 Entity Relationships
- `users` **(1)** → **(N)** `devices` — Each employee is assigned one or more devices.
- `users` **(1)** → **(N)** `log_events` — Events are traced back to the acting user.
- `devices` **(1)** → **(N)** `log_events` — Events originate from a specific device.
- `devices` **(1)** → **(N)** `alerts` — Alerts are triggered on a specific device.
- `alerts` **(1)** → **(1)** `response_actions` — Each alert receives exactly one triage response.
- `alerts` **(1)** → **(1)** `alert_outcomes` — Each alert has exactly one final verdict.

---

### 2.3 Table: `users` (Employee Directory & Behavioral Baselines)

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `user_id` | `string` (PK) | Unique employee identifier | `USR-0001`, `USR-0002` |
| 2 | `username` | `string` | Employee display name | `alice.chen`, `bob.martinez` |
| 3 | `role` | `categorical` | Access privilege level | `Standard_User`, `Admin`, `Privileged_User` |
| 4 | `department` | `categorical` | Organizational unit | `IT`, `Finance`, `HR`, `Engineering`, `Executive` |
| 5 | `hire_date` | `date` | Employment start date | `2019-03-15` |
| 6 | `typical_login_start` | `int` (hour) | Expected shift start hour | `7`, `8`, `9`, `22` (night shift) |
| 7 | `typical_login_end` | `int` (hour) | Expected shift end hour | `16`, `17`, `18`, `6` (night shift) |
| 8 | `vpn_flag` | `boolean` | Regular remote/VPN user | `True`, `False` |

---

### 2.4 Table: `devices` (Network Topology & Asset Criticality)

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `device_id` | `string` (PK) | Unique device identifier | `DEV-0001`, `DEV-0002` |
| 2 | `user_id` | `string` (FK) | Primary assigned user | `USR-0001` |
| 3 | `device_type` | `categorical` | Hardware category | `Laptop`, `Desktop`, `Server`, `Database_Server`, `IoT_Sensor` |
| 4 | `os_version` | `categorical` | Operating system | `Windows_11`, `Ubuntu_22.04`, `macOS_14`, `CentOS_7` |
| 5 | `criticality_score` | `int` | Asset importance (1=low, 10=critical) | `1`–`10` |
| 6 | `department` | `categorical` | Department (inherited from user) | `IT`, `Finance`, `HR`, `Engineering` |
| 7 | `ip_address` | `string` | Internal network IP | `10.0.1.45`, `192.168.5.12` |

### 2.5 Table: `log_events` (High-Volume Network Telemetry)

*This is the largest table — the raw, noisy, time-series data that the learner must mine for attack signals.*

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `event_id` | `string` (PK) | Unique event identifier | `EVT-0000001` |
| 2 | `timestamp` | `datetime` | Event time (with ±3s jitter) | `2026-06-01 14:23:07.412` |
| 3 | `device_id` | `string` (FK) | Source device | `DEV-0042` |
| 4 | `user_id` | `string` (FK) | Acting user | `USR-0015` |
| 5 | `event_type` | `categorical` | Type of network event | `Login_Success`, `Login_Fail`, `File_Download`, `File_Upload`, `Outbound_Connection`, `Process_Start`, `RDP_Session`, `Firewall_Block` |
| 6 | `source_ip` | `string` | Origin IP address (~3–5% missing) | `10.0.1.45`, `NaN` |
| 7 | `dest_ip` | `string` | Destination IP address | `10.0.2.100`, `203.0.113.50` |
| 8 | `bytes_transferred` | `float` | Data volume in bytes (Log-Normal) | `1024.0` (1 KB) to `3.2e8` (300 MB) |
| 9 | `geo_location` | `categorical` | Country of source IP | `PH`, `SG`, `US`, `DE`, `RU`, `CN` |
| 10 | `process_name` | `categorical` | Executable that triggered the event | See process list below |
| 11 | `session_duration_sec` | `float` | Duration of session/connection | `0.5`–`28800` (8 hours) |

#### Process Name Reference List

| Category | Process Names |
|:---|:---|
| **Normal IT** | `chrome.exe`, `outlook.exe`, `teams.exe`, `excel.exe`, `svchost.exe`, `explorer.exe`, `python.exe`, `code.exe`, `sqlservr.exe`, `nginx`, `sshd`, `systemd` |
| **Potentially Suspicious** | `powershell.exe`, `cmd.exe`, `wscript.exe`, `certutil.exe`, `curl.exe` |
| **Malicious (Red Team)** | `mimikatz.exe`, `psexec.exe`, `sharphound.exe`, `lazagne.exe`, `procdump.exe` |

> **Realism Note:** Malicious process names appear *almost exclusively* when `hidden_compromise_state = 1`. However, `powershell.exe` and `cmd.exe` also appear in ~15% of normal operations to prevent trivial detection.

---

### 2.6 Table: `alerts` (SIEM Aggregated Security Alerts)

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `alert_id` | `string` (PK) | Unique alert identifier | `ALR-00001` |
| 2 | `timestamp` | `datetime` | Alert generation time | `2026-06-01 14:25:00` |
| 3 | `device_id` | `string` (FK) | Device that triggered the alert | `DEV-0042` |
| 4 | `alert_rule_triggered` | `categorical` | Detection rule name | `Brute_Force`, `Data_Exfil_Suspected`, `Malware_Signature`, `Impossible_Travel`, `Unusual_Process`, `Port_Scan_Detected` |
| 5 | `severity` | `categorical` | Alert severity level | `Low`, `Medium`, `High`, `Critical` |
| 6 | `mitre_tactic` | `categorical` | MITRE ATT&CK tactic | `Reconnaissance`, `Initial_Access`, `Execution`, `Lateral_Movement`, `Exfiltration`, `Command_and_Control` |
| 7 | `confidence_score` | `float` | SIEM confidence (0.0 – 1.0) | `0.12` (noisy rule) to `0.95` (high confidence) |
| 8 | `alert_source` | `categorical` | Detection tool | `Firewall`, `EDR`, `IDS`, `Email_Gateway` |

---

### 2.7 Table: `response_actions` (SOC Analyst Triage)

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `action_id` | `string` (PK) | Unique action identifier | `ACT-00001` |
| 2 | `alert_id` | `string` (FK) | Associated alert | `ALR-00001` |
| 3 | `analyst_id` | `string` | SOC analyst who handled it | `SOC_T1_Alice`, `SOC_T2_Bob` |
| 4 | `action_taken` | `categorical` | Triage decision | `Isolate_Host`, `Block_IP`, `Reset_Credentials`, `Close_False_Positive`, `Escalate_to_Tier2` |
| 5 | `time_to_respond_mins` | `float` | Minutes to first action (Log-Normal, affected by fatigue) | `2.5`–`480.0` |
| 6 | `shift_time` | `categorical` | Analyst's current shift | `Day_Shift` (06:00–18:00), `Night_Shift` (18:00–06:00) |
| 7 | `alerts_handled_today` | `int` | Running count of alerts handled this shift | `1`–`100+` |

---

### 2.8 Table: `alert_outcomes` (Ground-Truth Labels & Business Impact)

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `alert_id` | `string` (PK/FK) | Associated alert | `ALR-00001` |
| 2 | `final_verdict` | `categorical` | Ground-truth classification | `True_Positive` (~3%), `False_Positive` (~95%), `Benign_True_Positive` (~2%) |
| 3 | `business_impact_usd` | `float` | Estimated financial impact | `$0` (FP) to `$4,500,000+` (major breach) |

> **Critical Note:** The `final_verdict` is *intentionally corrupted* by `hidden_analyst_fatigue`. Some True Positives are mislabeled as False Positives when the analyst's fatigue score exceeds 0.8. This simulates real-world label noise.

## 3. Realism & Messiness Injection Strategy

Real-world cybersecurity data is messy. The following artifacts are **intentionally injected** into the generated dataset to simulate production conditions:

### 3.1 Missing Values

| Table | Field | Missing Rate | Rationale |
|:---|:---|:---|:---|
| `log_events` | `source_ip` | ~3–5% | Firewalls under heavy load drop source IP metadata |
| `log_events` | `bytes_transferred` | ~1–2% | Some event types (e.g., `Login_Fail`) don't log payload size |
| `response_actions` | `time_to_respond_mins` | ~2% | Automated responses (scripts) don't record human response time |

### 3.2 Timestamp Jitter
- All `log_events.timestamp` values have **±3 seconds** of uniform random noise added.
- **Rationale:** Enterprise networks consist of hundreds of devices with imperfectly synchronized NTP clocks. Timestamps are never perfectly aligned in real SIEM data.

### 3.3 Class Imbalance
- `alert_outcomes.final_verdict` distribution:
  - **~95% False Positive** — The overwhelming majority of alerts are noise.
  - **~3% True Positive** — Actual breaches are rare.
  - **~2% Benign True Positive** — Real but harmless events (e.g., authorized pen tests).
- **Rationale:** Industry reports (Ponemon, 2020) confirm that SOCs face 95–99% false positive rates.

### 3.4 Duplicate Alerts
- ~5% of alerts are **near-duplicates** (same device, same rule, within ±60 seconds).
- **Rationale:** SIEMs often fire the same detection rule multiple times for a single underlying event.

### 3.5 Seasonal & Temporal Patterns
- **Monday mornings** see a spike in phishing-related alerts (employees returning from weekends).
- **Friday afternoons** see reduced analyst response quality (pre-weekend fatigue).
- **3:00 AM–5:00 AM** windows have elevated attack activity (adversaries prefer off-hours).
- **Rationale:** Both attacker behavior and defender fatigue follow predictable temporal patterns documented in threat intelligence reports.

### 3.6 Impossible Travel Injection
- During active compromise periods ($C_t = 1$), some `log_events` show the same `user_id` logging in from geographically impossible locations within short time windows (e.g., `PH` and `RU` within 5 minutes).
- **Rationale:** Impossible travel is a classic indicator of compromised credentials used simultaneously by the real user and the attacker.

## 4. Mathematical Model & Structural Equations

### 4.1 Hidden Compromise State ($C_t$) — The Puppet Master

The compromise state is a binary latent variable that governs the entire attack simulation. For each device $d$ in the network:

$$C_{d,t} \in \{0, 1\}$$

- $C_{d,t} = 0$: Device is operating normally (safe state).
- $C_{d,t} = 1$: Device has been compromised by an adversary.

**Transition Logic:** Compromise events are seeded at random times during the 30-day simulation window. Once a device is compromised, the compromise state persists for a **dwell time** $T_{dwell}$ drawn from:

$$T_{dwell} \sim \text{LogNormal}(\mu = 4.5, \; \sigma = 1.0) \quad \text{(in hours)}$$

> *Per the 2023 Mandiant M-Trends Report, the global median dwell time is ~16 days. We compress this to hours for a 30-day simulation window.*

---

### 4.2 Network Telemetry Volume — Poisson Process

The arrival rate of log events $\lambda_t$ for device $d$ at time $t$ is modeled as a **conditional Poisson process**:

$$\lambda_{d,t} = \lambda_{base} \cdot (1 + \gamma \cdot C_{d,t})$$

| Parameter | Value | Meaning |
|:---|:---|:---|
| $\lambda_{base}$ | 50 events/hour | Normal baseline traffic rate |
| $\gamma$ | 9 | Lateral movement multiplier |

**Interpretation:**
- **Safe state** ($C_{d,t} = 0$): $\lambda_{d,t} = 50 \cdot (1 + 9 \cdot 0) = 50$ events/hour
- **Compromised** ($C_{d,t} = 1$): $\lambda_{d,t} = 50 \cdot (1 + 9 \cdot 1) = 500$ events/hour

**Why Poisson?** Network events are discrete, non-negative counts that arrive randomly over time. The Poisson distribution naturally produces the bursty traffic patterns observed in real SIEM data.

---

### 4.3 Data Exfiltration Payloads — Log-Normal Distribution

The bytes transferred $B$ in each log event is modeled using a **Log-Normal distribution**, conditioned on the compromise state:

$$\ln(B) \sim \mathcal{N}(\mu_{C_t}, \; \sigma^2_{C_t})$$

| State | $\mu$ | $\sigma$ | Typical Value | Interpretation |
|:---|:---|:---|:---|:---|
| Safe ($C_t = 0$) | 5 | 1 | ~150 KB | Normal web browsing, email |
| Compromised ($C_t = 1$) | 15 | 2 | ~300+ MB | Database dumps, credential files |

**Why Log-Normal?** File sizes are heavily right-skewed: most transfers are small (emails, API calls), but exfiltration events involve massive data dumps. Log-Normal captures this heavy-tailed behavior while guaranteeing $B > 0$.

---

### 4.4 Human-in-the-Loop: Analyst Fatigue & Mislabeled Ground Truth

Let $H_i$ be the cumulative number of alerts handled by analyst $i$ during their current shift. The **latent fatigue score** $F_i \in [0, 1]$ is:

$$F_i = \min\left(1.0, \; \frac{H_i}{K}\right)$$

| Parameter | Value | Meaning |
|:---|:---|:---|
| $K$ | 50 | Fatigue threshold (alerts before max fatigue) |

**Impact on Response Time:**

$$\text{time\_to\_respond} \sim \text{LogNormal}(\mu_{base} + \beta_1 \cdot F_i, \; \sigma_{resp})$$

| Parameter | Value | Meaning |
|:---|:---|:---|
| $\mu_{base}$ | 2.0 | Base log-response time (~7 minutes) |
| $\beta_1$ | 1.5 | Fatigue penalty coefficient |
| $\sigma_{resp}$ | 0.8 | Response time variance |

**The "Omitted Variable" Trap — Label Corruption:**

If an alert is a True Breach ($C_t = 1$), but the analyst is fatigued ($F_i > 0$), the probability of **mislabeling** it as a False Positive increases:

$$P(\text{Label} = \text{FP} \mid \text{True State} = \text{TP}) = \alpha + \beta_2 \cdot F_i$$

| Parameter | Value | Meaning |
|:---|:---|:---|
| $\alpha$ | 0.05 | Base human error rate (5%) |
| $\beta_2$ | 0.75 | Fatigue penalty (max mislabel rate = 80%) |

**Interpretation at extremes:**
- Fresh analyst ($F_i = 0$): $P(\text{mislabel}) = 0.05 + 0.75 \times 0 = 5\%$
- Exhausted analyst ($F_i = 1$): $P(\text{mislabel}) = 0.05 + 0.75 \times 1 = 80\%$

> This means the learner's ML model will encounter **True Positive breaches labeled as False Positives** in the training data. Advanced learners can detect this by analyzing the rolling alert count per analyst and correlating it with suspicious verdict patterns.

## 5. Sampling / Generation Order

The generator must follow this **strict dependency order** to ensure referential integrity and statistical consistency:

| Step | Action | Depends On | Output |
|:---|:---|:---|:---|
| **1** | Generate `users` table | Config only (n_users, departments, roles) | `users.csv` |
| **2** | Generate `devices` table | `users` (FK: user_id) | `devices.csv` |
| **3** | Initialize `hidden_compromise_state` timeline | `devices`, Config (n_compromises, dwell_time params) | In-memory array $C_{d,t}$ |
| **4** | Generate `log_events` table | `devices`, `users`, $C_{d,t}$ (drives $\lambda_t$ and $\mu_{bytes}$) | `log_events.csv` |
| **5** | Aggregate `log_events` → Generate `alerts` table | `log_events`, detection rules, $C_{d,t}$ | `alerts.csv` |
| **6** | Generate `response_actions` + compute `hidden_analyst_fatigue` | `alerts`, analyst pool, shift schedule | `response_actions.csv` + in-memory $F_i$ |
| **7** | Generate `alert_outcomes` (with label corruption from $F_i$) | `alerts`, $C_{d,t}$, $F_i$ | `alert_outcomes.csv` |
| **8** | **Drop** all hidden/latent columns | — | Final clean export |

### Dependency Flow
```
Step 1: users
  │
  └──▶ Step 2: devices
          │
          └──▶ Step 3: hidden_compromise_state (LATENT — not exported)
                  │
                  └──▶ Step 4: log_events
                          │
                          └──▶ Step 5: alerts
                                  │
                                  ├──▶ Step 6: response_actions + hidden_analyst_fatigue (LATENT)
                                  │
                                  └──▶ Step 7: alert_outcomes (labels corrupted by fatigue)
                                          │
                                          └──▶ Step 8: DROP latent variables → EXPORT CSVs
```

## 6. Literature & Desk Research

The parameters and distributions chosen for this generator are grounded in industry cybersecurity reports and academic research:

### 6.1 Class Imbalance & False Positive Rates
According to a study by the Ponemon Institute (2020), the average enterprise SOC receives over 10,000 alerts daily, with nearly 45% of all alerts being false positives, and a significant portion of true breaches going entirely unnoticed due to alert volume. More specialized SOCs report false positive rates as high as 95–99%. This justifies our structural decision to set the True Positive rate at ~3%.

> Ponemon Institute. (2020). *The economics of security operations centers: What is the true cost for effective results?* Ponemon Institute LLC.

### 6.2 Attack Chains & Lateral Movement (MITRE ATT&CK)
The sequence of `mitre_tactic` variables and the 10× traffic multiplier ($\gamma = 9$) for compromised devices are based on the MITRE ATT&CK framework's documentation of enterprise breach lifecycles. Compromised credentials typically lead to rapid internal reconnaissance, lateral movement across network segments, and eventual data exfiltration.

> MITRE Corporation. (2023). *MITRE ATT&CK® framework: Enterprise matrix* (v13). https://attack.mitre.org/

### 6.3 Data Exfiltration Volumes & Breach Costs
The Verizon DBIR notes that the majority of breaches involve the exfiltration of large databases or credential stores, manifesting as heavy-tailed distributions in network egress traffic. This justifies our Log-Normal distribution ($\mu_1 = 15$, $\sigma_1 = 2$) for compromised states. The IBM Cost of a Data Breach Report (2022) reports an average breach cost of \$4.35M USD, informing our `business_impact_usd` distribution.

> Verizon Enterprise. (2023). *2023 data breach investigations report (DBIR)*. Verizon Communications.
>
> IBM Security. (2022). *Cost of a data breach report 2022*. IBM Corporation.

### 6.4 Analyst Fatigue & Human Error in SOCs
Research on SOC analyst burnout shows that cognitive load directly correlates with misclassification errors during triage. Studies indicate that after handling 40–60 alerts in a single shift, analyst accuracy degrades significantly. Our fatigue threshold ($K = 50$) and penalty coefficient ($\beta_2 = 0.75$) simulate this cognitive degradation.

> Sundaramurthy, S. C., McHugh, J., Ou, X., Wesch, M., Bardas, A. G., & Rajagopalan, S. R. (2016). Turning contradictions into innovations or: How we learned to stop whining and improve security operations. *Proceedings of the 12th Symposium on Usable Privacy and Security (SOUPS)*, 237–251.

### 6.5 Dwell Time
The 2023 Mandiant M-Trends report indicates a global median dwell time of 16 days for externally detected intrusions. Our compressed Log-Normal dwell time ($\mu = 4.5$ hours) is scaled proportionally for the 30-day simulation window.

> Mandiant. (2023). *M-Trends 2023: Special report*. Google Cloud / Mandiant.

## 7. Omitted Variables Declaration

To ensure the dataset requires genuine feature engineering and prevents trivial solutions, the following **latent variables** are generated in memory but are **strictly dropped** before the final CSV export:

---

### 7.1 `hidden_compromise_state` ($C_{d,t}$)

| Property | Detail |
|:---|:---|
| **Type** | Binary (0 = Safe, 1 = Compromised) |
| **Scope** | Per-device, per-hour time series |
| **Drives** | Log volume (Poisson $\lambda$), `bytes_transferred` distribution, malicious `process_name` probability, `alert` generation rate, `final_verdict` ground truth |

**Why it's omitted:** In the real world, a SOC analyst does not have a column that says "this device is hacked." The learner must infer compromise from secondary signals:
- Sudden spikes in log volume on a single device
- Anomalous `bytes_transferred` values (heavy tail)
- Appearance of red-team tools (`mimikatz.exe`, `psexec.exe`)
- Impossible travel patterns in `geo_location`
- Temporal clustering of high-severity alerts

---

### 7.2 `hidden_analyst_fatigue` ($F_i$)

| Property | Detail |
|:---|:---|
| **Type** | Continuous float [0.0, 1.0] |
| **Scope** | Per-analyst, per-shift |
| **Drives** | `time_to_respond_mins` inflation, `final_verdict` mislabeling probability |

**Why it's omitted:** Real-world datasets suffer from **label noise** due to human error, but the source of that noise is never explicitly recorded. By hiding the fatigue score, the learner's ML model will experience unexplained variance:
- "Why did my model miss this obvious breach?"
- "Why are there clusters of mislabeled alerts at certain times of day?"

**Detection strategies for advanced learners:**
- Rolling count of `alerts_handled_today` per `analyst_id`
- Interaction between `shift_time` (Night vs. Day) and `final_verdict` accuracy
- Time-of-day analysis on `time_to_respond_mins` distributions

---

### Summary of Omitted Variable Effects

| Omitted Variable | Observable Effect in Final Data | Detection Strategy |
|:---|:---|:---|
| `hidden_compromise_state` | Correlated spikes in log volume, bytes, malicious processes, and alerts on the same device within a time window | Device-level time-series aggregation + anomaly detection |
| `hidden_analyst_fatigue` | Clusters of mislabeled True Positives during night shifts or after high alert counts | Analyst-level rolling statistics + shift-based stratification |